In [0]:
def read_data(path, format="csv", schema=None, options=None):
    try:
        dbutils.fs.ls(path)
    except Exception:
        print(f"⏭️ Ruta no encontrada: {path}")
        if schema:
            return spark.createDataFrame([], schema)
        else:
            return spark.createDataFrame([], spark.createDataFrame([], []).schema)

    reader = spark.read

    if options:
        for k, v in options.items():
            reader = reader.option(k, v)

    if schema:
        reader = reader.schema(schema)

    if format == "csv":
        return reader.csv(path)
    
    elif format == "parquet":
        return reader.parquet(path)
    
    elif format == "json":
        return reader.json(path)
    
    elif format == "delta":
        return reader.format("delta").load(path)
    
    else:
        raise ValueError(f"Formato no soportado: {format}")

In [0]:
def read_csv(path, schema):
    try:
        dbutils.fs.ls(path)
    except Exception:
        print(f"Archivo no encontrado: {path}")
        return spark.createDataFrame([], schema)
    
    return spark.read \
        .option("header", True) \
        .option("sep", ",") \
        .schema(schema) \
        .csv(path)

In [0]:
from pyspark.sql.functions import current_timestamp

def add_ingestion_timestamp(input_df):
    output_df = input_df.withColumn("ingestion_timestamp", current_timestamp())
    return output_df

In [0]:
def handle_error(e, dbutils=None, max_len=1000):
    import traceback
    from datetime import datetime
    error_type = type(e).__name__
    error_summary = str(e)
    error_trace = traceback.format_exc()

    error_msg_full = f"""
        ERROR_TYPE: {error_type}
        ERROR_SUMMARY: {error_summary}
        TIMESTAMP: {datetime.now()}
        TRACEBACK:
        {error_trace}
        """

    # truncar si es muy largo
    if len(error_msg_full) > max_len:
        error_msg = error_msg_full[:max_len] + "\n[...] ERROR TRUNCADO [...]"
    else:
        error_msg = error_msg_full
    
    dbutils.jobs.taskValues.set(key="error", value=error_msg)
    return error_msg

In [0]:
def no_registros(df, columna, lista, tabla, fecha):
    from pyspark.sql.functions import col
    registros = df.filter(col(columna).isNull()).count()
    lista.append({
        "tabla":tabla,
        "columna":columna,
        "regla":"no_registros",
        "cumple":registros > 0,
        "detalle":f"No hay registros nuevos para la fecha: {fecha}"
    })

In [0]:
def nulos(df, columna, lista, tabla):
    from pyspark.sql.functions import col
    null = df.filter(col(columna).isNull()).count()
    lista.append({
        "tabla":tabla,
        "columna":columna,
        "regla":"no_null",
        "cumple":null == 0,
        "detalle":f"Registros nulos:{null}"
    })

In [0]:
def duplicados(df, columna, lista, tabla):
    from pyspark.sql.functions import col
    duplicates = df.groupBy(col(columna)).count().filter(col("count") > 1).count()
    lista.append({
        "tabla":tabla,
        "columna":columna,
        "regla":"no_duplicate",
        "cumple":duplicates == 0,
        "detalle":f"Registros duplicados:{duplicates}"
    })

In [0]:
def get_last_record(spark, table_name, timestamp_col="ingestion_timestamp"):
    query = f"""
    SELECT COALESCE(MAX({timestamp_col}), '1900-01-01')
    FROM {table_name}
    """
    
    return spark.sql(query).collect()[0][0]